In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/architandon/pth-files/models/best_model_swarm_2.pth
/kaggle/input/datasets/architandon/pth-files/models/best_model_swarm_1.pth
/kaggle/input/datasets/architandon/pth-files/models/best_model_swarm_0.pth
/kaggle/input/datasets/architandon/household-cluster-mapping/household_cluster_mapping.csv
/kaggle/input/datasets/architandon/src-load-forecast/feature_engineering.py
/kaggle/input/datasets/architandon/src-load-forecast/train.py
/kaggle/input/datasets/architandon/src-load-forecast/privacy_faults.py
/kaggle/input/datasets/architandon/src-load-forecast/evaluate.py
/kaggle/input/datasets/architandon/src-load-forecast/lstm_seq2seq.py
/kaggle/input/datasets/architandon/src-load-forecast/swarm_aggregator.py
/kaggle/input/datasets/architandon/src-load-forecast/dataset_loader.py
/kaggle/input/datasets/jeanmidev/smart-meters-in-london/darksky_parameters_documentation.html
/kaggle/input/datasets/jeanmidev/smart-meters-in-london/weather_hourly_darksky.csv
/kaggle/input/datase

In [4]:
import os
import sys
import gc
import random
import torch
import numpy as np
import pandas as pd

# 1. Point Python to your uploaded scripts
sys.path.append('/kaggle/input/datasets/architandon/src-load-forecast')
from lstm_seq2seq import VoltHiveLSTM
from dataset_loader import create_client_dataloaders

# CPU is perfectly fine for Evaluation!
DEVICE = torch.device("cpu")
SEQ_LEN = 48
FEATURES = 8

print("🚀 Booting up VoltHive Research Evaluation Suite...")

# ---------------------------------------------------------
# 2. LOCATE ALL DATA FILES DYNAMICALLY
# ---------------------------------------------------------
DATA_DIR = ""
MAPPING_PATH = ""
for root, dirs, files in os.walk('/kaggle/input'):
    if 'informations_households.csv' in files:
        DATA_DIR = root
    if 'household_cluster_mapping.csv' in files:
        MAPPING_PATH = os.path.join(root, 'household_cluster_mapping.csv')

if not DATA_DIR or not MAPPING_PATH:
    raise FileNotFoundError(" CRITICAL: Could not find Smart Meters dataset or Mapping CSV.")

print(" Loading Metadata...")
weather_df = pd.read_csv(os.path.join(DATA_DIR, "weather_hourly_darksky.csv"))
holidays_df = pd.read_csv(os.path.join(DATA_DIR, "uk_bank_holidays.csv"))
info_df = pd.read_csv(os.path.join(DATA_DIR, "informations_households.csv"))
mapping_df = pd.read_csv(MAPPING_PATH)

client_to_block_map = dict(zip(info_df['LCLid'], info_df['file']))

block_paths = {}
for root, dirs, files in os.walk('/kaggle/input'):
    if 'halfhourly' in root.lower():  
        for file in files:
            if file.startswith('block_') and file.endswith('.csv'):
                block_paths[file] = os.path.join(root, file)

# ---------------------------------------------------------
# 3. EVALUATION FUNCTIONS
# ---------------------------------------------------------
print("🔍 Hunting for .pth model files...")
discovered_models = {}
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.startswith('best_model_swarm_') and file.endswith('.pth'):
            # Extract the swarm ID from the filename
            s_id = int(file.split('_')[-1].split('.')[0])
            discovered_models[s_id] = os.path.join(root, file)

def load_swarm_model(swarm_id):
    if swarm_id not in discovered_models:
        print(f" Could not find model for Swarm {swarm_id} anywhere in /kaggle/input/")
        return None
        
    path = discovered_models[swarm_id]
    print(f"   [Loaded model from: {path}]")
    
    model = VoltHiveLSTM(input_features=FEATURES, output_steps=SEQ_LEN).to(DEVICE)
    # ---------------------------------------------------------
    # THE MULTI-GPU FIX
    # ---------------------------------------------------------
    # Load the raw weights
    raw_state_dict = torch.load(path, map_location=DEVICE)
    
    # Create a clean dictionary, stripping out the "module." prefix
    clean_state_dict = {}
    for key, value in raw_state_dict.items():
        clean_key = key.replace("module.", "")
        clean_state_dict[clean_key] = value
        
    # Load the cleaned weights
    model.load_state_dict(clean_state_dict)
    # ---------------------------------------------------------
    
    model.eval()
    return model

def calculate_metrics(actual, predicted):
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted)**2))
    return mae, rmse

# ---------------------------------------------------------
# 4. MAIN EVALUATION LOOP
# ---------------------------------------------------------
total_system_mae = []
total_system_rmse = []

# We will sample 40 households per swarm to evaluate. 
# (Evaluating all 4,000 would take hours, 40 gives a highly accurate statistical average)
SAMPLE_SIZE = 40 

for swarm_id in range(3):
    print(f"\n{'='*50}\n EVALUATING SWARM {swarm_id} EXPERT MODEL\n{'='*50}")
    
    model = load_swarm_model(swarm_id)
    if model is None: continue
        
    swarm_clients = mapping_df[mapping_df['cluster_id'] == swarm_id]['LCLid'].unique().tolist()
    # Randomly select a sample of households to test
    test_clients = random.sample(swarm_clients, min(SAMPLE_SIZE, len(swarm_clients)))
    
    swarm_maes = []
    swarm_rmses = []
    
    for client_id in test_clients:
        block_filename = client_to_block_map.get(client_id)
        if pd.isna(block_filename): continue
            
        block_filename = str(block_filename) + '.csv' if not str(block_filename).endswith('.csv') else str(block_filename)
        if block_filename not in block_paths: continue
            
        # Lazy Load Data
        block_path = block_paths[block_filename]
        temp_block_df = pd.read_csv(block_path, low_memory=False)
        raw_df = temp_block_df[temp_block_df['LCLid'] == client_id].copy()
        del temp_block_df
        gc.collect()
        
        if raw_df.empty: continue
            
        # Clean Data (Exact match to train.py)
        raw_df.columns = raw_df.columns.str.strip()
        rename_map = {'tstp': 'timestamp', 'energy(kWh/hh)': 'energy', 'energy(kwh/hh)': 'energy', 'Energy': 'energy'}
        raw_df = raw_df.rename(columns=rename_map)
        raw_df['energy'] = pd.to_numeric(raw_df['energy'], errors='coerce')
        raw_df = raw_df.dropna(subset=['energy']) 
        raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'])
        
        # Get the DataLoaders
        _, _, test_loader = create_client_dataloaders(raw_df, weather_df, holidays_df)
        
        if test_loader is None or len(test_loader) == 0: 
            continue
            
        # Run Inference
        client_maes = []
        client_rmses = []
        
        with torch.no_grad():
            for seq, target in test_loader:
                seq = seq.to(DEVICE)
                target = target.cpu().numpy()
                
                pred = model(seq).cpu().numpy()
                pred = np.maximum(pred, 0) # Zero out negative energy predictions
                
                mae, rmse = calculate_metrics(target, pred)
                client_maes.append(mae)
                client_rmses.append(rmse)
                
        if client_maes:
            swarm_maes.append(np.mean(client_maes))
            swarm_rmses.append(np.mean(client_rmses))
            
    # Calculate Final Swarm Metrics
    final_swarm_mae = np.mean(swarm_maes)
    final_swarm_rmse = np.mean(swarm_rmses)
    
    total_system_mae.append(final_swarm_mae)
    total_system_rmse.append(final_swarm_rmse)
    
    print(f" Swarm {swarm_id} Evaluated ({len(swarm_maes)} households tested)")
    print(f" Swarm {swarm_id} MAE:  {final_swarm_mae:.4f}")
    print(f" Swarm {swarm_id} RMSE: {final_swarm_rmse:.4f}")

# ---------------------------------------------------------
# 5. FINAL RESULTS
# ---------------------------------------------------------
print("\n" + "="*50)
print(" VOLTHIVE ARCHITECTURE: FINAL RESEARCH METRICS ")
print("="*50)
print(f"Overall System MAE:  {np.mean(total_system_mae):.4f}")
print(f"Overall System RMSE: {np.mean(total_system_rmse):.4f}")
print("="*50)

🚀 Booting up VoltHive Research Evaluation Suite...
 Loading Metadata...
🔍 Hunting for .pth model files...

 EVALUATING SWARM 0 EXPERT MODEL
   [Loaded model from: /kaggle/input/datasets/architandon/pth-files/models/best_model_swarm_0.pth]
 Swarm 0 Evaluated (40 households tested)
 Swarm 0 MAE:  0.3168
 Swarm 0 RMSE: 0.4612

 EVALUATING SWARM 1 EXPERT MODEL
   [Loaded model from: /kaggle/input/datasets/architandon/pth-files/models/best_model_swarm_1.pth]
 Swarm 1 Evaluated (40 households tested)
 Swarm 1 MAE:  0.1044
 Swarm 1 RMSE: 0.1648

 EVALUATING SWARM 2 EXPERT MODEL
   [Loaded model from: /kaggle/input/datasets/architandon/pth-files/models/best_model_swarm_2.pth]
 Swarm 2 Evaluated (40 households tested)
 Swarm 2 MAE:  0.1434
 Swarm 2 RMSE: 0.2055

 VOLTHIVE ARCHITECTURE: FINAL RESEARCH METRICS 
Overall System MAE:  0.1882
Overall System RMSE: 0.2772
